# MangoLeaf AI V5.1.0: raw and fixed 70% confidence evaluation

Run after the V5.1.0 training notebook completes. This unexecuted template reads that completed run's image snapshots, model, temperature record, and hash-checked handoff. It never retrains, refits temperature, searches rejection settings, or reads V5's old model weights/predictions.

**Raw** means the calibrated model's highest-scoring class before rejection. **70% confidence** uses those same predictions, accepting maximum probability >=0.70 and marking lower scores uncertain. Margin and entropy rejection thresholds are absent. Uncertain photos remain in per-class support and reduce recall. Accepted-only accuracy uses a different denominator from accuracy over all test photos.

**Manual steps:** select a fresh Colab Python 3.11/3.12 session. A T4 GPU is optional: select hardware accelerator **None** to evaluate on CPU. Run Step 1, restart the session, rerun Step 1, then continue. Step 2 probes and records the actual CPU/GPU device. Paste the exact completed training run into Step 3. No raw Drive folders or old V5 run folder are required here: training already copied the selected images. Do not rerun the output-creation or inference cell inside a started attempt.

**Shared training folder:** the owner may share the completed `v5.1.0` folder with the evaluating account using Viewer access. In that account, open Shared with me, select the folder, and choose Organize > Add shortcut > My Drive. Keep the shortcut named `v5.1.0` directly in My Drive. In Step 3, set `ALLOW_SHARED_INPUT_SHORTCUTS = True` and set `TRAINING_RUN` to `DRIVE_ROOT / 'v5.1.0/training/PASTE_EXACT_TRAINING_RUN_NAME'`, replacing only the placeholder with the completed run name. Authorize the Drive mount with the account that can see this shortcut. Sharing alone does not put the folder in the mounted My Drive. The full run must remain accessible, including `training_complete.json`, dataset/OE/reporting snapshots, model assets, and receipts; a model file alone is insufficient. These are read-only notebook inputs, so Viewer access is sufficient. No copy or retraining is needed. The notebook does not modify the source; new results go to the evaluating account's own `MyDrive/mangoleaf_v5_1_0_evaluations/<unique-run>/`. Do not make that output folder a shared-source shortcut. Resource availability and account usage remain subject to Colab's rules.

CPU evaluation uses the same saved model, class order, preprocessing, and fixed cutoff; no weights or calibration are refitted. Execution can be slower. CPU/GPU numerical kernels can produce small score differences, which may change close class ties or decisions very near 70%; record the measured results and actual device rather than claiming bitwise hardware equivalence. This notebook still requires a fully completed training handoff, not an interrupted training attempt.

Outputs include raw/70% confusion matrices and per-class figures, overall comparison, unsupported-image chart, full-precision JSON/NumPy/CSV measurements, a candidate TensorFlow.js copy, and an evaluation receipt. The old 397 mango test photos and 995 unsupported photos have prior evaluation use. Report the historical mango subset separately from any new test additions; do not describe reused collections as a fresh independent field test.

## Step 1: Mount Drive and install the candidate runtime

**What:** Prepare the compatible inference runtime.

**Why:** Evaluation must use recorded dependency versions rather than mixed session imports.

**How:** Install binary wheels, restart the session, then rerun this cell before continuing.

**Expected result:** Setup receipt and logs; no predictions.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

# Check Python before mounting Drive or installing scientific packages.
print('Notebook Python:', sys.version.split()[0])
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(
        'Unsupported Python ' + sys.version.split()[0] + '. '
        'Open Runtime > Change runtime type, keep a GPU such as T4, and select '
        'Runtime Version 2026.04 (Python 3.12.13), then rerun Step 1. '
        'If unavailable, use a listed Python 3.11/3.12 runtime: '
        'https://research.google.com/colaboratory/runtime-version-faq.html. '
        'These TensorFlow 2.19.0 pins must not run on an unsupported Python version.'
    )

from google.colab import drive

drive.mount('/content/drive')

TRAINING_PACKAGES = [
    'tensorflow==2.19.0',
    'keras==3.15.1',
    'numpy==2.1.3',
    'Pillow==13.0.0',
    'ImageHash==4.3.2',
    'scipy==1.16.3',
    'matplotlib==3.11.2',
    'scikit-learn==1.9.1',
    'pip-audit==2.10.1',
    'packaging==25.0',
    'uv==0.13.4',
    # Explicit security pins prevent pip from retaining vulnerable provider helpers.
    'pip==26.2.1',
    'setuptools==84.1.0',
    'certifi==2026.10.18',
    'filelock==4.2.1',
    'idna==3.15',
    'msgpack==1.2.3',
    'requests==2.33.0',
    'urllib3==2.8.0',
    'werkzeug==3.1.9',
]
setup_directory = Path('/content/mangoleaf-v510-setup-20261101')
setup_directory.mkdir(exist_ok=True)
setup_receipt = setup_directory / 'installation.json'
installed = json.loads(setup_receipt.read_text()) if setup_receipt.is_file() else None

if installed is None:
    # Upgrade the installer first; this subprocess does not import scientific modules.
    pip_pin = next(pin for pin in TRAINING_PACKAGES if pin.startswith('pip=='))
    bootstrap = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         '--only-binary=:all:', '--no-deps', pip_pin],
        capture_output=True, text=True, check=False,
    )
    (setup_directory / 'pip-bootstrap.log').write_text(bootstrap.stdout + '\n' + bootstrap.stderr)
    if bootstrap.returncode != 0:
        raise RuntimeError('Installer update failed. Read: ' + str(setup_directory / 'pip-bootstrap.log'))
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         '--only-binary=:all:', *TRAINING_PACKAGES],
        capture_output=True, text=True, check=False,
    )
    (setup_directory / 'installation.log').write_text(result.stdout + '\n' + result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Package installation failed. Read: ' + str(setup_directory / 'installation.log'))
    setup_receipt.write_text(json.dumps({
        'packages': TRAINING_PACKAGES, 'installationPid': os.getpid(),
        'python': sys.version, 'restartRequired': True,
    }, indent=2) + '\n')
    print('Installation complete. Restart session, then rerun Step 1 and continue.')
else:
    if installed['packages'] != TRAINING_PACKAGES:
        raise RuntimeError('Setup pins changed. Use a fresh runtime rather than mixing installation attempts.')
    print('Packages already installed. Restart completed:', installed['installationPid'] != os.getpid())

## Step 2: Audit dependencies and verify the runtime

**What:** Check versions, dependencies, and CPU/GPU execution.

**Why:** Successful import alone does not establish a complete working environment.

**How:** Run the same fresh research audit; use a GPU when available or a CPU otherwise. Probe and record the selected device.

**Expected result:** Runtime evidence with the actual device, or a blocking error.

In [ ]:
import csv
import hashlib
import importlib.metadata
import math
import random
import shutil
import uuid
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import PurePosixPath

# Stop before scientific imports can mix pre-install modules with new files.
installed = json.loads(setup_receipt.read_text())
if installed['installationPid'] == os.getpid():
    raise RuntimeError('Restart the Colab session after installation, rerun Step 1, then rerun this cell.')

import imagehash
import matplotlib.pyplot as plt
import numpy as np
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from PIL import Image, ImageDraw, ImageOps
from scipy.optimize import minimize_scalar
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

for pin in TRAINING_PACKAGES:
    name, expected = pin.split('==')
    if importlib.metadata.version(name) != expected:
        raise RuntimeError('Runtime version differs from chosen pin: ' + pin)
if tf.__version__ != '2.19.0' or tf.keras.__version__ != '3.15.1':
    raise RuntimeError('Imported TensorFlow/Keras versions differ from installed pins; restart a clean session.')

# Audit only the transitive research runtime, rather than treating every
# unrelated preinstalled Colab provider package as an executed experiment dependency.
def installed_research_dependencies(pins):
    """Collect the actual transitive research dependency closure."""
    names = set()
    examined_contexts = set()
    pending_requirements = [Requirement(pin) for pin in pins]
    while pending_requirements:
        requirement = pending_requirements.pop()
        name = canonicalize_name(requirement.name)
        distribution = importlib.metadata.distribution(name)
        if not requirement.specifier.contains(distribution.version, prereleases=True):
            raise RuntimeError(
                'Research dependency conflict: ' + str(requirement)
                + '; installed ' + distribution.version
            )
        names.add(name)
        # Extras such as CacheControl[filecache] add dependencies that must be audited.
        # Revisit a package when a later requirement enables a different extra.
        for extra in {''} | requirement.extras:
            context = (name, extra)
            if context in examined_contexts:
                continue
            examined_contexts.add(context)
            for dependency in distribution.requires or []:
                child = Requirement(dependency)
                if child.marker is None or child.marker.evaluate({'extra': extra}):
                    pending_requirements.append(child)
    return names


research_names = installed_research_dependencies(TRAINING_PACKAGES)

runtime_lock = setup_directory / 'training-runtime-lock.txt'
runtime_lock.write_text(''.join(
    name + '==' + importlib.metadata.version(name) + '\n'
    for name in sorted(research_names)
), encoding='utf-8')
audit_path = setup_directory / 'training-audit.json'
audit = subprocess.run([
    sys.executable, '-m', 'pip_audit', '-r', str(runtime_lock),
    '--no-deps', '--disable-pip', '--format=json', '--output', str(audit_path),
    '--progress-spinner=off',
], capture_output=True, text=True, check=False)
(setup_directory / 'training-audit.log').write_text(audit.stdout + '\n' + audit.stderr)
audit_report = json.loads(audit_path.read_text()) if audit_path.is_file() else {}
dependencies = audit_report.get('dependencies', [])
findings = [row for row in dependencies if row.get('vulns')]
skipped = [row for row in dependencies if row.get('skip_reason')]
audited_names = {canonicalize_name(row['name']) for row in dependencies if row.get('name')}
if audit.returncode != 0 or findings or skipped or audited_names != research_names:
    print('Dependency audit exit:', audit.returncode)
    for row in findings:
        print('Unresolved package:', row['name'], row.get('version', 'unknown'))
        details = sorted({
            vulnerability.get('id', 'unknown') + ' (fix versions: '
            + ', '.join(vulnerability.get('fix_versions', [])) + ')'
            for vulnerability in row['vulns']
        })
        for detail in details:
            print('  ', detail)
    for row in skipped:
        print('Skipped package:', row.get('name', 'unknown'), row['skip_reason'])
    print('Missing packages:', sorted(research_names - audited_names))
    print('Unexpected packages:', sorted(audited_names - research_names))
    print('Full audit and log:', audit_path, setup_directory / 'training-audit.log')
    raise RuntimeError('Training dependency audit incomplete or unresolved. Read ' + str(audit_path))

provider_check = subprocess.run(
    [sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True, check=False,
)
(setup_directory / 'provider-pip-check.log').write_text(provider_check.stdout + '\n' + provider_check.stderr)
if provider_check.returncode:
    print('Colab provider compatibility findings remain visible:', setup_directory / 'provider-pip-check.log')


gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
EVALUATION_DEVICE = '/GPU:0' if gpus else '/CPU:0'
with tf.device(EVALUATION_DEVICE):
    execution_probe = tf.linalg.matmul(tf.ones((16, 16)), tf.ones((16, 16)))
if not execution_probe.device.endswith(EVALUATION_DEVICE.lstrip('/')) or not np.isfinite(execution_probe.numpy()).all():
    raise RuntimeError('The selected evaluation device execution probe failed.')
RUNTIME = {
    'python': sys.version, 'tensorflow': tf.__version__,
    'packages': {name: importlib.metadata.version(name) for name in sorted(research_names)},
    'trainingDependencyAuditPassed': True, 'providerPipCheckReturnCode': provider_check.returncode,
    'gpuCount': len(gpus), 'evaluationDevice': EVALUATION_DEVICE,
    'executionProbeDevice': execution_probe.device,
    'gpuProbeDevice': execution_probe.device if gpus else None,
    'execution': 'ordinary Colab Python cells; CPU evaluation supported; no weight updates',
    'browserParityPassed': False,
}
print('TensorFlow:', tf.__version__, 'Keras:', tf.keras.__version__, 'Evaluation device:', EVALUATION_DEVICE)
print('CPU evaluation is slower in general. Tiny hardware-dependent score differences are possible; do not assume bitwise GPU parity.')

## Step 3: Select and verify the completed training run

**What:** Read the exact V5.1.0 training receipt and image snapshots.

**Why:** Mixing models, class orders, or image lists invalidates a comparison.

**How:** Set TRAINING_RUN to the completed run. For a shared input, add its shortcut to My Drive and enable ALLOW_SHARED_INPUT_SHORTCUTS. Verify every completed artifact and create a separate owned output directory under mangoleaf_v5_1_0_evaluations.

**Expected result:** Verified model, mango test rows, 995 reporting photos, and a unique evaluation directory.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
# Copy the exact path printed at the end of the training notebook.
TRAINING_RUN = DRIVE_ROOT / 'mangoleaf_experiments/v5.1.0/training/PASTE_EXACT_TRAINING_RUN_NAME'
# For a shared v5.1.0 shortcut placed directly in My Drive, use:
# TRAINING_RUN = DRIVE_ROOT / 'v5.1.0/training/PASTE_EXACT_TRAINING_RUN_NAME'
ALLOW_SHARED_INPUT_SHORTCUTS = False  # Set True only for your selected shared training input.
# Keep results outside the shared source folder, in this mounted account's My Drive.
OUTPUT_ROOT = DRIVE_ROOT / 'mangoleaf_v5_1_0_evaluations'
EXPERIMENT_VERSION = 'v5.1.0'
CLASS_NAMES = ['anthracnose', 'bacterial_black_spot', 'bacterial_canker',
               'healthy_fruit', 'healthy_leaf', 'mango_scab', 'sooty_mold']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CONFIDENCE_THRESHOLD = 0.70

def drive_input_path(path, allow_shared=False, my_drive=Path('/content/drive/MyDrive')):
    """Accept selected My Drive inputs, with explicit shared-shortcut permission."""
    selected = Path(path)
    my_drive = Path(my_drive)
    if (not selected.is_absolute() or '..' in selected.parts
            or selected == my_drive or not selected.is_relative_to(my_drive)):
        raise ValueError('Select an input folder through a path below My Drive: ' + str(selected))

    resolved = selected.resolve()
    owned_root = my_drive.resolve()
    shortcut_root = (my_drive.parent / '.shortcut-targets-by-id').resolve()
    owned = resolved != owned_root and resolved.is_relative_to(owned_root)
    shared = (resolved.is_relative_to(shortcut_root)
              and len(resolved.relative_to(shortcut_root).parts) >= 2)
    print('Input:', selected, '\nResolved:', resolved)
    if not owned and not (allow_shared and shared):
        raise ValueError(
            'Input resolves outside My Drive. For a reviewed shared input, add its shortcut '
            'to My Drive and set ALLOW_SHARED_INPUT_SHORTCUTS = True. '
            'Other external paths remain unsupported: ' + str(resolved)
        )
    if not resolved.is_dir():
        raise FileNotFoundError('Input folder missing or inaccessible in the mounted account: ' + str(selected))
    return selected


def owned_my_drive_path(path, my_drive=Path('/content/drive/MyDrive')):
    """Keep evaluation outputs in the mounted account's own My Drive."""
    selected = Path(path)
    resolved = selected.resolve()
    owned_root = Path(my_drive).resolve()
    print('Selected location:', selected)
    print('Resolved location:', resolved)
    print('Exists:', resolved.exists())
    if resolved == owned_root or not resolved.is_relative_to(owned_root):
        raise ValueError(
            'Evaluation output paths must resolve inside this mounted account\'s '
            'My Drive. Select an output folder owned by this account, not a shortcut '
            'to another account\'s output folder: ' + str(resolved)
        )
    return resolved


def file_sha256(path):
    """Hash file bytes without loading the whole image into memory."""
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def object_sha256(value):
    """Identify structured configuration with explicit canonical JSON."""
    serialized = json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False)
    return hashlib.sha256(serialized.encode('utf-8')).hexdigest()


def read_json(path):
    """Read recorded UTF-8 JSON without altering its bytes."""
    return json.loads(Path(path).read_text(encoding='utf-8'))


def save_json_once(path, value):
    """Write evidence once and reject a changed overwrite."""
    path = Path(path)
    if path.exists():
        if read_json(path) != value:
            raise RuntimeError('Refusing to overwrite changed evidence: ' + str(path))
        return
    with path.open('x', encoding='utf-8', newline='\n') as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write('\n')


def safe_child(root, relative):
    """Resolve a validated relative path inside its declared root."""
    if not isinstance(relative, str) or not relative or '\\' in relative or ':' in relative or '\x00' in relative:
        raise ValueError('Unsafe relative path: ' + str(relative))
    child = PurePosixPath(relative)
    if child.is_absolute() or '..' in child.parts or child == PurePosixPath('.'):
        raise ValueError('Unsafe relative path: ' + relative)
    destination = Path(root).joinpath(*child.parts).resolve()
    destination.relative_to(Path(root).resolve())
    return destination


def raw_rgb(path):
    """Decode and orient original image pixels for identity checks."""
    with Image.open(path) as opened:
        return ImageOps.exif_transpose(opened).convert('RGB')


def load_pixels(path):
    # TensorFlow matches the app's bilinear float32 input. Pillow handles WebP
    # content, including WebP files whose suffix claims JPEG.
    """Match the recorded 224x224 bilinear model preprocessing."""
    with Path(path).open('rb') as handle:
        signature = handle.read(12)
    if signature[:4] == b'RIFF' and signature[8:12] == b'WEBP':
        image = tf.convert_to_tensor(np.asarray(raw_rgb(path)), dtype=tf.uint8)
    else:
        image = tf.io.decode_image(tf.io.read_file(str(path)), channels=3, expand_animations=False)
    return tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE, method='bilinear', antialias=False)


def portable_rows(rows):
    """Remove machine-local paths from portable image records."""
    return [{key: value for key, value in row.items() if key != 'path'} for row in rows]


def verify_image_rows(rows):
    """Detect changed photo bytes before fitting or reporting."""
    for row in rows:
        if file_sha256(row['path']) != row['sha256']:
            raise RuntimeError('Input image changed after inventory: ' + row['path'])


if 'PASTE_EXACT' in str(TRAINING_RUN):
    raise ValueError('Set TRAINING_RUN to the completed V5.1.0 run before continuing.')
TRAINING_RUN_SELECTED = TRAINING_RUN
# Shared snapshots are inputs only; every generated file stays below OUTPUT_ROOT.
TRAINING_RUN = drive_input_path(TRAINING_RUN, allow_shared=ALLOW_SHARED_INPUT_SHORTCUTS).resolve()
OUTPUT_ROOT = owned_my_drive_path(OUTPUT_ROOT)
if OUTPUT_ROOT == TRAINING_RUN or OUTPUT_ROOT.is_relative_to(TRAINING_RUN) or TRAINING_RUN.is_relative_to(OUTPUT_ROOT):
    raise ValueError('Evaluation output and completed training input must be separate.')
COMPLETE_PATH = TRAINING_RUN / 'training_complete.json'
COMPLETE_HASH = file_sha256(COMPLETE_PATH)
complete = read_json(COMPLETE_PATH)
if complete.get('experimentVersion') != EXPERIMENT_VERSION or complete.get('completedEpochs') != 15:
    raise RuntimeError('Select a completed fifteen-epoch V5.1.0 training run.')
if complete.get('classNames') != CLASS_NAMES or not complete.get('temperatureFoldedIntoWeights'):
    raise RuntimeError('Training class order or calibration declaration differs.')
for relative, expected in complete['artifacts'].items():
    path = safe_child(TRAINING_RUN, relative)
    if path.stat().st_size != expected['bytes'] or file_sha256(path) != expected['sha256']:
        raise RuntimeError('Completed training artifact changed: ' + relative)
training_plan = read_json(TRAINING_RUN / 'plan.json')
if training_plan.get('confidenceThreshold') != CONFIDENCE_THRESHOLD:
    raise RuntimeError('Evaluation confidence cutoff differs from the pre-training plan.')
manifest = read_json(TRAINING_RUN / 'input_manifest.json')
if manifest['classNames'] != CLASS_NAMES or manifest['experimentVersion'] != EXPERIMENT_VERSION:
    raise RuntimeError('Training manifest version or class order differs.')
rows = [{**row, 'path': str(safe_child(TRAINING_RUN / 'dataset', row['relativePath']))} for row in manifest['images']]
oe_rows = [{**row, 'path': str(safe_child(TRAINING_RUN, row['relativePath']))}
           for row in read_json(TRAINING_RUN / 'oe_dataset_manifest.json')['images']]
ood_rows = [{**row, 'path': str(safe_child(TRAINING_RUN, row['relativePath']))}
            for row in read_json(TRAINING_RUN / 'reporting_ood_manifest.json')['images']]
if len(oe_rows) != 2511 or len(ood_rows) != 995:
    raise RuntimeError('OE/reporting membership differs from the completed design.')
test_rows = [row for row in rows if row['subset'] == 'test']
if not test_rows or any(row['subset'] not in ('train', 'val', 'test') for row in rows):
    raise RuntimeError('The mango split contains missing or invalid membership.')
reporting_reference = read_json(TRAINING_RUN / 'reference-inputs/v5-evaluation-reporting-lock.json')['images']
if {r['sha256'] for r in ood_rows} != {r['sha256'] for r in reporting_reference}:
    raise RuntimeError('Reporting membership does not match the 995 locked identities.')

run_id = 'v5-1-0-evaluation-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
FIGURE_DIR = RUN_DIR / 'figures'
FIGURE_DIR.mkdir()
shutil.copytree(setup_directory, RUN_DIR / 'runtime_preflight')
save_json_once(RUN_DIR / 'runtime.json', RUNTIME)
save_json_once(RUN_DIR / 'training_reference.json', {'trainingRun': str(TRAINING_RUN),
    'selectedTrainingRun': str(TRAINING_RUN_SELECTED), 'trainingInputMode': 'read-only',
    'allowSharedInputShortcuts': ALLOW_SHARED_INPUT_SHORTCUTS,
    'trainingCompleteSha256': COMPLETE_HASH, 'calibratedModelSha256': file_sha256(TRAINING_RUN / 'model_calibrated.keras')})
print('Verified completed training run:', TRAINING_RUN)
print('Mango test count:', len(test_rows), 'reporting OOD:', len(ood_rows))
print('Evaluation output:', RUN_DIR)

## Step 4: Freeze the fixed-confidence reporting plan

**What:** Check leakage boundaries and seal inputs before predictions.

**Why:** Test or unsupported reporting scores must never select calibration or a rejection cutoff.

**How:** Check exact identities and whole-group membership; lock calibrated confidence >=0.70.

**Expected result:** Hashed evaluation inputs and reporting plan; no margin/entropy settings.

In [ ]:
def assert_disjoint_pools(pools):
    """Reject exact file/pixel identity leakage across scientific pools."""
    for index, (name, first) in enumerate(pools):
        for field in ('sha256', 'decodedRgbSha256'):
            values = [row[field] for row in first]
            if len(values) != len(set(values)):
                raise RuntimeError('Duplicate ' + field + ' inside ' + name)
            for second_name, second in pools[index + 1:]:
                if set(values) & {row[field] for row in second}:
                    raise RuntimeError('Exact overlap between ' + name + ' and ' + second_name)




# Repeat cheap split/pool checks on the actual sealed evaluation inputs.
assert_disjoint_pools([('mango', rows), ('OE', oe_rows), ('reporting OOD', ood_rows)])
group_subsets = defaultdict(set)
for row in rows:
    group_subsets[row['groupId']].add(row['subset'])
if any(len(subsets) != 1 for subsets in group_subsets.values()):
    raise RuntimeError('A related mango group crosses training/validation/test.')
for selected in (rows, oe_rows, ood_rows):
    verify_image_rows(selected)
evaluation_inputs = {
    'mangoTest': portable_rows(test_rows), 'reportingOod': portable_rows(ood_rows),
    'referenceCount': 995, 'trainingCompleteSha256': COMPLETE_HASH,
    'classNames': CLASS_NAMES, 'confidenceThreshold': CONFIDENCE_THRESHOLD,
    'priorUse': 'Old mango test and 995 OOD photos already evaluated in earlier experiments.',
    'selectionPolicy': 'No evaluation images or predictions choose calibration or the fixed confidence cutoff.',
}
save_json_once(RUN_DIR / 'evaluation_inputs.json', evaluation_inputs)
INPUT_HASH = file_sha256(RUN_DIR / 'evaluation_inputs.json')
reporting_plan = {'experimentVersion': EXPERIMENT_VERSION, 'trainingCompleteSha256': COMPLETE_HASH,
    'modelSha256': file_sha256(TRAINING_RUN / 'model_calibrated.keras'), 'evaluationInputsSha256': INPUT_HASH,
    'classNames': list(CLASS_NAMES), 'confidenceThreshold': CONFIDENCE_THRESHOLD, 'rule': 'max calibrated probability >= 0.70',
    'rawMeaning': 'Same calibrated model highest-scoring predictions before rejection',
    'temperatureAppliedAgain': False, 'modelSelectionUsesTest': False,
    'evaluationDevice': EVALUATION_DEVICE, 'runtimeSha256': file_sha256(RUN_DIR / 'runtime.json')}
save_json_once(RUN_DIR / 'reporting_plan.json', reporting_plan)
REPORTING_PLAN_HASH = file_sha256(RUN_DIR / 'reporting_plan.json')
print('Fixed reporting plan sealed before model predictions:', REPORTING_PLAN_HASH)

## Step 5: Define one prediction pass and two reporting views

**What:** Validate score vectors and derive before/after decisions.

**Why:** Raw and thresholded figures must describe the same model and same photos.

**How:** Require seven normalized finite probabilities per photo. Use argmax for raw and maximum probability >=0.70 for acceptance.

**Expected result:** Strict scoring helpers bound to the sealed model and plan.

In [ ]:
def image_dataset(rows, labelled=False):
    """Stream one original photo at a time in manifest order."""
    def examples():
        """Yield original image pixels one at a time."""
        for row in rows:
            pixels = load_pixels(row['path']).numpy().astype(np.float32)
            if labelled:
                yield pixels, np.int32(CLASS_NAMES.index(row['label']))
            else:
                yield pixels
    signature = tf.TensorSpec((224, 224, 3), tf.float32)
    if labelled:
        signature = (signature, tf.TensorSpec((), tf.int32))
    dataset = tf.data.Dataset.from_generator(examples, output_signature=signature)
    return dataset.apply(tf.data.experimental.assert_cardinality(len(rows)))



def validate_probabilities(values, expected_count):
    """Reject malformed scores rather than hiding invalid model outputs."""
    if len(values) != expected_count:
        raise ValueError('Expected one probability vector per photo.')
    for vector in values:
        if len(vector) != 7:
            raise ValueError('Expected seven class probabilities per photo.')
        if any(not math.isfinite(float(value)) or not 0 <= float(value) <= 1 for value in vector):
            raise ValueError('Scores must be finite probabilities between zero and one.')
        if not math.isclose(sum(float(value) for value in vector), 1.0, rel_tol=0, abs_tol=1e-5):
            raise ValueError('Scores are not normalized probabilities.')
    return values


def score_predictions(records, probabilities, labelled):
    """Describe raw and confidence-only decisions from the same score vectors."""
    values = validate_probabilities(probabilities, len(records))
    scored = []
    for row, vector in zip(records, values):
        index = int(np.argmax(vector))
        confidence = float(vector[index])
        keep = confidence >= CONFIDENCE_THRESHOLD
        scored.append({'id': row['id'], 'sha256': row['sha256'], 'pool': row['pool'],
            'origin': row.get('origin', row['pool']),
            'actualClass': row['label'] if labelled else None,
            'rawPredictedClass': CLASS_NAMES[index], 'confidence': confidence,
            'accepted': keep, 'displayedClass': CLASS_NAMES[index] if keep else 'rejected'})
    return scored


def verify_reporting_inputs():
    """Keep every result bound to one completed model and fixed reporting plan."""
    if file_sha256(COMPLETE_PATH) != COMPLETE_HASH:
        raise RuntimeError('Completed training receipt changed.')
    if file_sha256(RUN_DIR / 'evaluation_inputs.json') != INPUT_HASH:
        raise RuntimeError('Evaluation identities changed.')
    if file_sha256(RUN_DIR / 'reporting_plan.json') != REPORTING_PLAN_HASH:
        raise RuntimeError('Fixed reporting plan changed.')
    if reporting_plan['modelSha256'] != file_sha256(TRAINING_RUN / 'model_calibrated.keras'):
        raise RuntimeError('Calibrated model changed.')
    if reporting_plan['confidenceThreshold'] != CONFIDENCE_THRESHOLD or CONFIDENCE_THRESHOLD != 0.70:
        raise RuntimeError('Only the predeclared fixed 70% confidence rule is permitted.')
    if reporting_plan['evaluationDevice'] != EVALUATION_DEVICE or reporting_plan['runtimeSha256'] != file_sha256(RUN_DIR / 'runtime.json'):
        raise RuntimeError('Evaluation device or recorded runtime changed after plan freeze.')
    if reporting_plan['classNames'] != CLASS_NAMES:
        raise RuntimeError('Live class order differs from the frozen plan.')
    if portable_rows(test_rows) != evaluation_inputs['mangoTest'] or portable_rows(ood_rows) != evaluation_inputs['reportingOod']:
        raise RuntimeError('Live evaluation membership/order differs from the frozen inputs.')


verify_reporting_inputs()

## Step 6: Evaluate mango and unsupported reporting snapshots

**What:** Run new inference and calculate both result sets.

**Why:** Old V4.5.0/V5.0.0 prediction values cannot be reused as V5.1.0 results.

**How:** Load only this run's calibrated model; predict once per photo. Keep uncertain photos in all-photo and per-class denominators.

**Expected result:** Probabilities, predictions, raw/70% metrics, and separate historical/additional mango summaries.

In [ ]:
def classification_metrics(actual, predicted):
    """Count each class including rejected true-class photos as misses."""
    if len(actual) != len(predicted):
        raise ValueError('Every true class must have exactly one predicted result.')
    if any(label not in CLASS_NAMES for label in actual):
        raise ValueError('Actual mango labels must belong to the seven supported classes.')
    if any(label not in CLASS_NAMES + ['rejected'] for label in predicted):
        raise ValueError('Predicted results must be supported classes or the rejection result.')
    metrics = []
    for label in CLASS_NAMES:
        true_positive = sum(truth == label and guess == label for truth, guess in zip(actual, predicted))
        false_positive = sum(truth != label and guess == label for truth, guess in zip(actual, predicted))
        false_negative = sum(truth == label and guess != label for truth, guess in zip(actual, predicted))
        support = sum(truth == label for truth in actual)
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        metrics.append({
            'className': label, 'precision': precision, 'recall': recall, 'f1Score': f1,
            'support': support, 'truePositive': true_positive,
            'falsePositive': false_positive, 'falseNegative': false_negative,
        })
    return metrics


def summarize_mango(scored):
    """Keep raw, all-photo, and accepted-only denominators explicit."""
    if not scored:
        raise ValueError('A mango reporting subset cannot be empty.')
    actual = [row['actualClass'] for row in scored]
    raw_predicted = [row['rawPredictedClass'] for row in scored]
    selected_predicted = [row['displayedClass'] for row in scored]
    accepted_count = sum(row['accepted'] for row in scored)
    correct_raw = sum(truth == predicted for truth, predicted in zip(actual, raw_predicted))
    correct_accepted = sum(row['accepted'] and row['actualClass'] == row['rawPredictedClass'] for row in scored)
    raw_matrix = [[sum(truth == first and predicted == second for truth, predicted in zip(actual, raw_predicted)) for second in CLASS_NAMES] for first in CLASS_NAMES]
    rejection_matrix = [[sum(truth == first and predicted == second for truth, predicted in zip(actual, selected_predicted)) for second in CLASS_NAMES + ['rejected']] for first in CLASS_NAMES]
    return {
        'total': len(scored), 'accepted': accepted_count, 'rejected': len(scored) - accepted_count,
        'retention': accepted_count / len(scored), 'rawCorrect': correct_raw, 'rawAccuracy': correct_raw / len(scored),
        'correctAccepted': correct_accepted, 'incorrectAccepted': accepted_count - correct_accepted,
        'correctAcceptedFractionOfAllPhotos': correct_accepted / len(scored),
        'acceptedOnlyAccuracy': correct_accepted / accepted_count if accepted_count else None,
        'rawPerClassMetrics': classification_metrics(actual, raw_predicted),
        'rejectionAwarePerClassMetrics': classification_metrics(actual, selected_predicted),
        'rawConfusionMatrix': raw_matrix, 'rejectionAwareConfusionMatrix': rejection_matrix,
        'matrixRowLabels': CLASS_NAMES, 'rejectionMatrixColumnLabels': CLASS_NAMES + ['rejected'],
        'denominatorNote': 'All true mango photos remain in support. Rejected true mango photos count as missed class predictions.',
    }


def summarize_ood(scored):
    """Count unsupported false acceptances and uncertain rejections."""
    if not scored:
        raise ValueError('An OOD reporting subset cannot be empty.')
    false_accepts = sum(row['accepted'] for row in scored)
    return {
        'total': len(scored), 'falseAccepted': false_accepts, 'rejected': len(scored) - false_accepts,
        'falseAcceptRate': false_accepts / len(scored),
        'rejectionRate': (len(scored) - false_accepts) / len(scored),
    }



verify_reporting_inputs()
if (RUN_DIR / 'evaluation_started.json').exists():
    raise RuntimeError('This attempt already started evaluation. Preserve it; do not overwrite or rerun inference in place.')
for selected in (test_rows, ood_rows):
    verify_image_rows(selected)
with tf.device(EVALUATION_DEVICE):
    model = tf.keras.models.load_model(TRAINING_RUN / 'model_calibrated.keras', compile=False, safe_mode=True)
    if tuple(model.input_shape[1:]) != (224, 224, 3) or model.output_shape[-1] != 7:
        raise RuntimeError('The sealed inference model has unexpected input/output dimensions.')
    save_json_once(RUN_DIR / 'evaluation_started.json', {
        'startedAt': datetime.now(timezone.utc).isoformat(), 'reportingPlanSha256': REPORTING_PLAN_HASH})

    # The model already contains temperature calibration. Do not rescale probabilities again.
    mango_probabilities = validate_probabilities(model.predict(image_dataset(test_rows).batch(BATCH_SIZE).prefetch(1), verbose=1), len(test_rows))
    ood_probabilities = validate_probabilities(model.predict(image_dataset(ood_rows).batch(BATCH_SIZE).prefetch(1), verbose=1), len(ood_rows))
np.save(RUN_DIR / 'mango_test_probabilities.npy', mango_probabilities)
np.save(RUN_DIR / 'ood_probabilities.npy', ood_probabilities)
mango_scored = score_predictions(test_rows, mango_probabilities, labelled=True)
ood_scored = score_predictions(ood_rows, ood_probabilities, labelled=False)
save_json_once(RUN_DIR / 'mango_predictions.json', mango_scored)
save_json_once(RUN_DIR / 'ood_predictions.json', ood_scored)

# Keep the historical mango test and any newly added test photos identifiable.
original_hashes = {row['sha256'] for row in read_json(TRAINING_RUN / 'reference-inputs/v5-training-input-manifest.json')['mango']}
historical_scored = [row for row in mango_scored if row['sha256'] in original_hashes]
added_scored = [row for row in mango_scored if row['sha256'] not in original_hashes]
evaluation = {
    'experimentVersion': EXPERIMENT_VERSION, 'confidenceThreshold': CONFIDENCE_THRESHOLD,
    'mangoTest': summarize_mango(mango_scored), 'unsupportedReporting': summarize_ood(ood_scored),
    'historicalMangoTest': summarize_mango(historical_scored) if historical_scored else None,
    'additionalMangoTest': summarize_mango(added_scored) if added_scored else None,
    'rawMeaning': 'Highest-scoring class before rejection from the calibrated model',
    'limits': 'Recorded comparisons; old test and OOD inputs have prior evaluation use; additional labels are not independently verified by hashes.',
    'trainingCompleteSha256': COMPLETE_HASH, 'reportingPlanSha256': REPORTING_PLAN_HASH,
    'evaluationDevice': EVALUATION_DEVICE, 'runtimeSha256': reporting_plan['runtimeSha256'],
}
save_json_once(RUN_DIR / 'evaluation.json', evaluation)
EVALUATION_HASH = file_sha256(RUN_DIR / 'evaluation.json')
verify_reporting_inputs()
for selected in (test_rows, ood_rows):
    verify_image_rows(selected)
print(json.dumps({key: value for key, value in evaluation['mangoTest'].items()
                  if key in ('total', 'rawCorrect', 'rawAccuracy', 'accepted', 'rejected', 'correctAccepted', 'incorrectAccepted',
                             'correctAcceptedFractionOfAllPhotos', 'acceptedOnlyAccuracy')}, indent=2))
print('Unsupported reporting:', evaluation['unsupportedReporting'])

## Step 7: Generate raw and 70% confidence figures

**What:** Draw figures from the measured evaluation JSON.

**Why:** Different accuracy denominators and rejected photos must remain visible.

**How:** Plot count/row-percent confusion matrices, precision/recall/F1, overall measures, and the 995-photo unsupported outcomes.

**Expected result:** Six PNG figures plus captions, all labeled V5.1.0 and based on actual scores.

In [ ]:
verify_reporting_inputs()
if file_sha256(RUN_DIR / 'evaluation.json') != EVALUATION_HASH:
    raise RuntimeError('Measured results changed before plotting.')
from IPython.display import Markdown, display
labels = [label.replace('_', ' ').title() for label in CLASS_NAMES]
captions = {}

def finish_figure(figure, filename, caption):
    """Save a measured figure with its interpretation caption."""
    figure.tight_layout(rect=(0, 0.04, 1, 0.96))
    figure.text(0.5, 0.01, caption, ha='center', va='bottom', fontsize=9, wrap=True)
    figure.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(figure)
    captions[filename] = caption
    display(Markdown('**' + filename + ':** ' + caption))

def plot_matrix_pair(matrix, column_labels, title, filename, caption):
    """Show class counts and row percentages with identical membership."""
    values = np.asarray(matrix, dtype=int)
    row_totals = values.sum(axis=1, keepdims=True)
    percentages = np.divide(values * 100.0, row_totals,
                            out=np.zeros_like(values, dtype=float), where=row_totals != 0)
    figure, axes = plt.subplots(1, 2, figsize=(22, 9))
    for axis, panel, panel_title, as_percent in [
        (axes[0], values, 'Counts', False),
        (axes[1], percentages, 'Row percentages (recall on diagonal)', True),
    ]:
        image = axis.imshow(panel, cmap='Blues', vmin=0, vmax=100 if as_percent else None)
        figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
        contrast = 50 if as_percent else max(1, float(panel.max()) / 2)
        for row in range(panel.shape[0]):
            for column in range(panel.shape[1]):
                value = panel[row, column]
                text = f'{value:.1f}%' if as_percent else str(int(value))
                axis.text(column, row, text, ha='center', va='center', fontsize=9,
                          color='white' if value > contrast else 'black')
        axis.set_xticks(range(len(column_labels)), column_labels, rotation=45, ha='right')
        axis.set_yticks(range(len(labels)), labels)
        axis.set_xlabel('Predicted result')
        axis.set_ylabel('Actual class')
        axis.set_title(panel_title)
    figure.suptitle(title, fontsize=16)
    finish_figure(figure, filename, caption)

mango = evaluation['mangoTest']
plot_matrix_pair(
    mango['rawConfusionMatrix'], labels,
    f"V5.1.0 confusion matrix on held-out mango test (n = {mango['total']})",
    'raw_confusion_matrix.png',
    'Raw seven-class predictions before rejection. Rows are actual classes; columns are predicted classes. '
    'Each right-panel row sums to 100%; its diagonal is class recall. Calibrated model before rejection; recorded test evidence with prior-use limits.',
)
plot_matrix_pair(
    mango['rejectionAwareConfusionMatrix'], labels + ['Uncertain'],
    f"V5.1.0 confusion matrix with the fixed 70% confidence rule (n = {mango['total']})",
    'confidence_70_confusion_matrix.png',
    'All true mango test photos remain in the denominator. The Uncertain column shows missed diagnoses; '
    'diagonal row percentages are rejection-aware recall. Rejection is a gate, not an eighth trained class.',
)

def plot_class_metrics(rows, overall_score, overall_label, title, filename, caption):
    """Plot precision, recall, and F1 with actual class support."""
    locations = np.arange(len(rows))
    figure, axis = plt.subplots(figsize=(17, 8))
    for offset, field, color, label in [
        (-0.26, 'precision', '#1f77b4', 'Precision'),
        (0.00, 'recall', '#ff7f0e', 'Recall'),
        (0.26, 'f1Score', '#2ca02c', 'F1-score'),
    ]:
        values = [row[field] for row in rows]
        bars = axis.bar(locations + offset, values, width=0.25, color=color, label=label)
        axis.bar_label(bars, labels=[f'{value:.2f}' for value in values], padding=3, fontsize=9)
    axis.axhline(overall_score, color='gray', linestyle='--',
                 label=f'{overall_label} {100 * overall_score:.2f}%')
    axis.set_xticks(locations, [label + '\n(n=' + str(row['support']) + ')'
                               for label, row in zip(labels, rows)], rotation=15, ha='right')
    axis.set_ylim(0, 1.12)
    axis.set_ylabel('Score (1.0 = 100%)')
    axis.set_title(title)
    axis.grid(axis='y', alpha=0.2)
    axis.legend(loc='lower right')
    finish_figure(figure, filename, caption)

plot_class_metrics(
    mango['rawPerClassMetrics'], mango['rawAccuracy'], 'Overall raw accuracy',
    'V5.1.0 per-class precision, recall, and F1 on held-out mango test',
    'raw_per_class_metrics.png',
    'Blue precision: how often a predicted class is correct. Orange recall: how many real class photos were found. '
    'Green F1 balances both. Support n is the actual class count; bar labels are rounded for display only.',
)
plot_class_metrics(
    mango['rejectionAwarePerClassMetrics'], mango['correctAcceptedFractionOfAllPhotos'],
    'Correct accepted / all mango photos',
    'V5.1.0 per-class metrics with the fixed 70% confidence rule',
    'confidence_70_per_class_metrics.png',
    'All true mango test photos remain in class support. Rejected mango photos count as missed class predictions. '
    'The dashed line is correct accepted predictions divided by all test photos, not accepted-only accuracy.',
)



figure, axis = plt.subplots(figsize=(11, 6))
summary_names = ['Raw correct / all', 'Correct accepted / all', 'Correct / accepted']
summary_values = [mango['rawAccuracy'], mango['correctAcceptedFractionOfAllPhotos'], mango['acceptedOnlyAccuracy']]
for index, (name, value) in enumerate(zip(summary_names, summary_values)):
    if value is not None:
        bar = axis.bar(index, value * 100, color=['#1f77b4', '#2ca02c', '#9467bd'][index])
        axis.bar_label(bar, labels=[f'{100 * value:.2f}%'], padding=3)
    else:
        axis.text(index, 2, 'Undefined: no accepted photos', ha='center', rotation=90)
axis.set_xticks(range(3), summary_names)
axis.set_ylim(0, 110)
axis.set_ylabel('Photographs (%)')
axis.set_title('V5.1.0 mango results: denominators differ')
finish_figure(figure, 'mango_overall_comparison.png',
    'Raw accuracy uses all test photos. Correct accepted/all counts uncertain photos as not correctly accepted. '
    'Accepted-only accuracy excludes uncertain photos; it must be read with the accepted and rejected counts.')

unsupported = evaluation['unsupportedReporting']
figure, axis = plt.subplots(figsize=(9, 6))
bars = axis.bar(['Rejected as uncertain', 'Accepted as a mango class'],
               [unsupported['rejected'], unsupported['falseAccepted']], color=['#2ca02c', '#d62728'])
axis.bar_label(bars, labels=[str(unsupported['rejected']), str(unsupported['falseAccepted'])], padding=3)
axis.set_ylabel('Number of unsupported photographs')
axis.set_title('V5.1.0 unsupported-image results at 70% confidence (n=995)')
axis.set_ylim(0, max(unsupported['rejected'], unsupported['falseAccepted'], 1) * 1.15)
finish_figure(figure, 'unsupported_confidence_70.png',
    'The 995 locked photos were previously evaluated in V5. Accepted unsupported photos are false acceptances; '
    'this collection does not have seven-class mango ground-truth labels.')
save_json_once(FIGURE_DIR / 'figure_captions.json', captions)

## Step 8: Export tables and the candidate model copy

**What:** Save full-precision class/overall measurements and candidate inference assets.

**Why:** Readable figures must not replace the underlying evidence or alter trained weights.

**How:** Write raw/70% CSV tables and copy model assets with only the fixed confidence reporting rule added to metadata.

**Expected result:** CSV files and an experimental TensorFlow.js copy; no application adoption.

In [ ]:
def write_csv_once(path, records, fields):
    """Export complete-precision measurements without overwriting evidence."""
    with Path(path).open('x', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(records)


verify_reporting_inputs()
for name, values in [('raw', evaluation['mangoTest']['rawPerClassMetrics']),
                     ('confidence_70', evaluation['mangoTest']['rejectionAwarePerClassMetrics'])]:
    write_csv_once(RUN_DIR / (name + '_per_class_metrics.csv'), values,
                   ['className', 'support', 'precision', 'recall', 'f1Score', 'truePositive', 'falsePositive', 'falseNegative'])
mango = evaluation['mangoTest']
overall = [{'result': 'correctAccepted', 'count': mango['correctAccepted'], 'fractionOfAll': mango['correctAccepted'] / mango['total']},
           {'result': 'incorrectAccepted', 'count': mango['incorrectAccepted'], 'fractionOfAll': mango['incorrectAccepted'] / mango['total']},
           {'result': 'uncertain', 'count': mango['rejected'], 'fractionOfAll': mango['rejected'] / mango['total']}]
write_csv_once(RUN_DIR / 'confidence_70_overall_results.csv', overall, ['result', 'count', 'fractionOfAll'])
candidate_directory = RUN_DIR / 'tfjs_model'
shutil.copytree(TRAINING_RUN / 'tfjs_model', candidate_directory)
candidate_configuration_path = candidate_directory / 'model_config.json'
configuration = read_json(candidate_configuration_path)
configuration.update({'status': 'experimental candidate; fixed-confidence evaluation complete; browser parity/adoption pending',
                      'confidenceThreshold': 0.70, 'rejectionRule': 'maximum calibrated probability >= 0.70 only'})
candidate_configuration_path.write_text(json.dumps(configuration, indent=2, allow_nan=False) + '\n', encoding='utf-8')
graph = read_json(candidate_directory / 'model.json')
for group in graph['weightsManifest']:
    for relative in group['paths']:
        if file_sha256(safe_child(candidate_directory, relative)) != file_sha256(safe_child(TRAINING_RUN / 'tfjs_model', relative)):
            raise RuntimeError('Model weights changed during candidate packaging.')
print('Candidate model copy:', candidate_directory, '(not installed or adopted)')

## Step 9: Seal evaluation and preserve its limits

**What:** Hash the completed evaluation outputs.

**Why:** Completed numerical comparisons still do not establish independent field accuracy or phone reliability.

**How:** Verify model/input/report identities again and write an immutable completion receipt.

**Expected result:** A completed evaluation receipt, with browser parity, physical evaluation, and adoption still pending.

In [ ]:
verify_reporting_inputs()
if file_sha256(RUN_DIR / 'evaluation.json') != EVALUATION_HASH:
    raise RuntimeError('Evaluation measurements changed before sealing.')
required = [path for path in sorted(RUN_DIR.rglob('*')) if path.is_file() and path.name != 'evaluation_complete.json']
save_json_once(RUN_DIR / 'evaluation_complete.json', {
    'experimentVersion': EXPERIMENT_VERSION, 'completedAt': datetime.now(timezone.utc).isoformat(),
    'trainingCompleteSha256': COMPLETE_HASH, 'evaluationSha256': EVALUATION_HASH,
    'confidenceThreshold': CONFIDENCE_THRESHOLD, 'trainingRun': str(TRAINING_RUN),
    'mangoTestCount': len(test_rows), 'unsupportedReportingCount': len(ood_rows),
    'browserParityPassed': False, 'physicalPhoneEvaluationPassed': False, 'adopted': False,
    'evaluationDevice': EVALUATION_DEVICE, 'runtimeSha256': reporting_plan['runtimeSha256'],
    'artifacts': {path.relative_to(RUN_DIR).as_posix(): {'sha256': file_sha256(path), 'bytes': path.stat().st_size} for path in required},
    'limits': evaluation['limits'],
})
print('Evaluation complete:', RUN_DIR)
print('Save this notebook with its actual outputs and preserve the full run folder.')
print('Raw results and 70% results share the same calibrated predictions. No margin or entropy rejection was used.')